In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import cvxpy as cp
from scipy.special import eval_hermitenorm
from math import factorial

from time_series.data_handlers import TimeSeriesData
from time_series.models import EigenGausRascutti

2026-03-17 10:05:53.298 | INFO     | time_series.config:<module>:13 - PROJ_ROOT path is: /home/james/Repo/PhD Repo/time_series_clustering


In [2]:
def create_dataset(theta, n_points, n_correlated_dimensions, n_uncorrelated_dimensions, noise = 0):
    n_dim = n_correlated_dimensions + n_uncorrelated_dimensions

    # Generate transition matrix
    T = np.zeros((n_dim, n_dim))
    np.fill_diagonal(T, np.cos(theta))

    for i in range(n_correlated_dimensions - 1):
        T[i, i+1] = np.sin(theta)

    for i in range(n_correlated_dimensions - 1):
        T[i+1, i] = -np.sin(theta)

    T[1, 2] = 0 # Decouple first two dimensions from rest

    # Generate data
    xi = np.random.random(n_dim)
    x = [xi]
    for i in range(n_points):
        xi = T@xi + np.random.normal(0, noise, size=(n_dim))
        x.append(xi)

    return np.stack(x)

In [21]:
ref_data = create_dataset(
    2,
    n_points=1000,
    n_correlated_dimensions=3,
    n_uncorrelated_dimensions=0,
    noise=0.2,
)

ref_dataset = TimeSeriesData(
    X=ref_data[:-1],
    y=ref_data[1:],
    lag=1,
    train_val_test_split=[0.5, 0.5],
)

In [22]:
X_train, y_train = ref_dataset.train_data(squeeze=True)
X_val, y_val = ref_dataset.val_data(squeeze=True)

In [24]:
model = EigenGausRascutti(1)
model.fit(X_train, y_train)

/home/james/Repo/PhD Repo/time_series_clustering/venv/lib/python3.12/site-packages/cvxpy/problems/problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


In [25]:
y_pred = model.predict(X_val)

In [27]:
np.mean((y_train - model.predict(X_train))**2)

np.float64(8.907481539687971)

In [28]:
np.mean((y_val - model.predict(X_val))**2)

np.float64(79.43829792886793)